#Connection Pool เต็ม

In [5]:
import time
import csv
from urllib.parse import urlparse
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

START_URL = "https://www.tooktee.com/Property/Location?StatusId=0"
OUTPUT_FILE = "tooktee_listing_urls.csv"
TIMEOUT = 40
MAX_PAGES = 30

def wait_ready(driver):
    t0 = time.time()
    while time.time() - t0 < TIMEOUT:
        if driver.execute_script("return document.readyState") == "complete":
            return True
        time.sleep(0.2)
    return False


def extract_links(driver, base):
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll("a[href^='/project/']"))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)

    out = []
    for h in hrefs:
        out.append(base + h if h.startswith("/") else h)

    return list(dict.fromkeys(out))

def scroll(driver):
    last_h = 0
    for _ in range(10):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.8)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == last_h:
            break
        last_h = h


def click_next(driver):
    try:
        el = driver.find_element(By.XPATH, "//li[contains(@class,'page-item next')]")
        if "disabled" in el.get_attribute("class"):
            return False

        btn = el.find_element(By.TAG_NAME, "button")
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", btn)
        return True
    except:
        return False

def main():
    opt = uc.ChromeOptions()
    opt.add_argument("--no-sandbox")
    opt.add_argument("--disable-dev-shm-usage")
    opt.add_argument("--disable-gpu")
    opt.add_argument("--disable-notifications")
    opt.add_argument("--disable-blink-features=AutomationControlled")

    driver = uc.Chrome(options=opt, version_main=145)

    base = f"{urlparse(START_URL).scheme}://{urlparse(START_URL).netloc}"

    driver.get(START_URL)
    wait_ready(driver)
    time.sleep(3)

    page = 1
    urls = set()

    print(f"Loading initial URL: {START_URL}")

    while page <= MAX_PAGES:
        print(f"[Page {page}/{MAX_PAGES}] Scrolling down and collecting links...")

        scroll(driver)
        found = extract_links(driver, base)

        if found:
            urls.update(found)
            print(f"  -> Found {len(found)} listings (total: {len(urls)})")
        else:
            print("  -> Found 0 listings")

        if not click_next(driver):
            print("  -> Stop pagination (no next / disabled / error)")
            break

        print(f"[Page {page}] Next clicked")
        time.sleep(4)
        wait_ready(driver)
        page += 1

    driver.quit()

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["ListingURL"])
        for u in sorted(urls):
            w.writerow([u])

    print(f"\nDone. total {len(urls)} saved -> {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Loading initial URL: https://www.tooktee.com/Property/Location?StatusId=0
[Page 1/30] Scrolling down and collecting links...
  -> Found 0 listings
  -> Stop pagination (no next / disabled / error)

Done. total 0 saved -> tooktee_listing_urls.csv


In [ ]:
import os
import csv
import time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

BASE_DIR = '/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/tooktee'
INPUT_CSV = os.path.join(BASE_DIR, 'tooktee_listing_urls.csv')
OUTPUT_CSV = os.path.join(BASE_DIR, 'tooktee_full_details.csv')

def extract_content(driver):
    full_content = []

    try:
        title_element = driver.find_element(By.ID, 'cssPropertyTitle')
        title_text = title_element.text.strip()
        if title_text:
            full_content.append("--- Listing Title ---")
            full_content.append(title_text)
    except Exception:
        pass

    try:
        right_info = driver.find_element(By.CSS_SELECTOR, '.right-info-box')
        right_text = right_info.text.strip()
        if right_text:
            full_content.append("\n--- Price and Contact Information ---")
            full_content.append(right_text)

        try:
            phone_element = driver.find_element(By.ID, 'MemberPhoneNum')
            full_phone = phone_element.get_attribute('data-num')
            if full_phone:
                full_content.append(f"Phone Number (Full): {full_phone}")
        except Exception:
            pass

        try:
            line_element = driver.find_element(By.ID, 'MemberLineID')
            full_line = line_element.get_attribute('data-lineid')
            if full_line:
                full_content.append(f"Line ID: {full_line}")
        except Exception:
            pass

    except Exception:
        pass

    try:
        project_info = driver.find_element(By.ID, 'projectinfo')

        try:
            driver.execute_script(
                "document.getElementById('propDescription').classList.remove('expired');"
            )
        except Exception:
            pass

        info_text = project_info.text.strip()
        if info_text:
            full_content.append("\n--- Property Details ---")
            full_content.append(info_text)
    except Exception:
        pass

    return "\n".join(full_content)

def main():
    os.makedirs(BASE_DIR, exist_ok=True)

    urls = []

    if os.path.exists(INPUT_CSV):
        with open(INPUT_CSV, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if row and len(row) > 0:
                    urls.append(row[0])
    else:
        print(f"File not found {INPUT_CSV}, using sample URL for testing")
        urls = [
            "https://www.tooktee.com/project/2611/1/191899/"
        ]

    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")

    driver = uc.Chrome(options=options, version_main=145)

    print(f"Starting to scrape total {len(urls)} items...")

    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Post_URL', 'Full_Content'])

        for i, url in enumerate(urls, 1):
            print(f"[{i}/{len(urls)}] Scraping: {url}")
            try:
                driver.get(url)
                time.sleep(3)

                content = extract_content(driver)
                writer.writerow([url, content])

            except Exception as e:
                print(f"Error scraping {url}: {e}")

    driver.quit()
    print(f"\nScraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()